# Stage 02 — Human-in-the-loop Taxonomy Review

Reads `source_patents.xlsx` (written by `01_review.ipynb` / `src.reviewer.run_stage01()`), shows one patent at a time across three panes — Patent Document, Aircraft Display, Taxonomy Characterization — pre-filled from the ML predictions, and appends your corrections to `reviewed_patents.xlsx` on **Save & Next**.

Missing/zero figures fall back to a bundled "No Image Available" placeholder — the image pane never raises.

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
repo_root = None
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / "src").exists() and (_candidate / "config.yaml").exists():
        repo_root = _candidate
        break
if repo_root is None:
    raise RuntimeError(f"Cannot find repo root from {_cwd}. Run from inside Patent-Labelling-Tools.")

for p in [str(repo_root), str(repo_root / "src")]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))
print(f"repo_root : {repo_root}")

In [ ]:
import datetime as dt
import getpass

import pandas as pd
import ipywidgets as W
from IPython.display import display

from src.config_loader import load_config
from src.reviewer import PLACEHOLDER_IMAGE_PATH
from src.excel_schema import COLUMNS, append_reviewed_rows

cfg = load_config()
SOURCE_XLSX   = Path(cfg["paths"]["source_excel"])
REVIEWED_XLSX = Path(cfg["paths"]["reviewed_excel"])
REVIEWER_NAME = getpass.getuser()

if not SOURCE_XLSX.exists():
    raise FileNotFoundError(f"{SOURCE_XLSX} not found — run 01_review.ipynb's run_stage01() first.")

df = pd.read_excel(SOURCE_XLSX, sheet_name="Review")
df = df.reindex(columns=COLUMNS)

# Already-reviewed patents start from their saved (corrected) rows, not the raw ML output.
if REVIEWED_XLSX.exists():
    reviewed_df = pd.read_excel(REVIEWED_XLSX, sheet_name="Review")
    done_ids = set(reviewed_df["Patent_ID"].unique())
    for pid in done_ids:
        df = df[df["Patent_ID"] != pid]
    df = pd.concat([df, reviewed_df.reindex(columns=COLUMNS)], ignore_index=True)
else:
    done_ids = set()

patent_ids = list(dict.fromkeys(df["Patent_ID"].tolist()))
print(f"Loaded {len(patent_ids)} patents ({len(done_ids)} already reviewed) from {SOURCE_XLSX.name}")

In [ ]:
BOOL_OPTIONS = {"true", "false"}

def _conf_color(conf):
    if conf is None or (isinstance(conf, float) and pd.isna(conf)):
        return "#888"  # grey — human-only, no ML prediction
    conf = float(conf)
    if conf >= 0.7:
        return "#1a7f37"  # green
    if conf >= 0.4:
        return "#9a6700"  # yellow/amber
    return "#cf222e"  # red


def load_image_bytes(path) -> bytes:
    """Read image bytes, falling back to the bundled placeholder on any failure."""
    try:
        if path:
            p = Path(path)
            if p.exists():
                return p.read_bytes()
    except Exception:
        pass
    return PLACEHOLDER_IMAGE_PATH.read_bytes()


def make_value_widget(row: pd.Series) -> W.Widget:
    options = [o for o in str(row["Options"] or "").split("|") if o]
    value = row["Value"]
    value = None if (value is None or (isinstance(value, float) and pd.isna(value))) else value

    if row["Field"] == "parts" and options:
        selected = [v for v in str(value or "").split("|") if v]
        return W.SelectMultiple(options=options, value=tuple(v for v in selected if v in options),
                                 rows=min(len(options), 6), layout=W.Layout(width="320px"))
    if options and set(o.lower() for o in options) <= BOOL_OPTIONS:
        return W.Checkbox(value=str(value).lower() == "true", indent=False)
    if options:
        opts = [""] + options
        return W.Dropdown(options=opts, value=value if value in opts else "", layout=W.Layout(width="320px"))
    return W.Text(value="" if value is None else str(value), layout=W.Layout(width="320px"))


def widget_value_to_str(widget: W.Widget) -> str:
    v = widget.value
    if isinstance(widget, W.SelectMultiple):
        return "|".join(v)
    if isinstance(widget, W.Checkbox):
        return "true" if v else "false"
    return "" if v is None else str(v)

In [ ]:
current_idx = 0
row_widgets: dict[int, W.Widget] = {}     # df row-index -> live widget, for the currently built form
current_image_paths: list[str] = []
current_image_idx = 0

doc_pane    = W.HTML()
image_pane  = W.Image(format="png", layout=W.Layout(width="380px", height="280px", object_fit="contain"))
image_label = W.HTML()
img_prev_btn = W.Button(description="◀ Prev image")
img_next_btn = W.Button(description="Next image ▶")
accordion   = W.Accordion()
status_bar  = W.HTML()

save_btn = W.Button(description="Save & Next", button_style="success")
skip_btn = W.Button(description="Skip", button_style="")
back_btn = W.Button(description="◀ Back", button_style="")

def sync_widgets_to_df():
    """Write the currently-displayed widgets' values back into df before navigating away."""
    for idx, widget in row_widgets.items():
        df.at[idx, "Value"] = widget_value_to_str(widget)


def render_image():
    if not current_image_paths:
        image_pane.value = load_image_bytes(None)
        image_label.value = "<i>No figures for this patent</i>"
        return
    path = current_image_paths[current_image_idx]
    image_pane.value = load_image_bytes(path)
    image_label.value = (f"<b>{Path(path).name if path else '(none)'}</b> "
                          f"— image {current_image_idx + 1}/{len(current_image_paths)}")

In [ ]:
def build_form(patent_id: str):
    global row_widgets, current_image_paths, current_image_idx
    row_widgets = {}
    current_image_idx = 0

    sub = df[df["Patent_ID"] == patent_id]

    # ── Left pane: Patent Document Viewer ──────────────────────────────────
    t1 = sub[sub["Section"] == "T1"]
    def _t1(field):
        m = t1[t1["Field"] == field]
        return "" if m.empty or pd.isna(m.iloc[0]["Value"]) else str(m.iloc[0]["Value"])
    doc_html = (
        f"<h3>{_t1('title') or patent_id}</h3>"
        f"<p><b>Assignee:</b> {_t1('assignee')} &nbsp; "
        f"<b>Pub year:</b> {_t1('pub_year')} &nbsp; <b>App year:</b> {_t1('app_year')}</p>"
        f"<p><b>Abstract:</b><br>{_t1('abstract')}</p>"
        f"<p><b>Description of drawings:</b><br>{_t1('description_of_drawings')}</p>"
    )
    doc_pane.value = doc_html

    # ── Middle pane: distinct figures for this patent ──────────────────────
    t2 = sub[sub["Section"] == "T2"]
    current_image_paths = list(dict.fromkeys(t2["Image_Path"].tolist())) or [str(PLACEHOLDER_IMAGE_PATH)]
    render_image()

    # ── Right pane: Accordion of Section -> widgets ─────────────────────────
    sections = ["T1", "T2", "G1", "M1", "M2", "M3"]
    children = []
    for section in sections:
        sec_df = sub[sub["Section"] == section]
        if section == "T2":
            # Only show rows for the figure currently on screen.
            cur_path = current_image_paths[current_image_idx]
            sub_dims = t2[t2["Image_Path"] == cur_path]["Sub_Dimension"].unique().tolist()
            sec_df = sec_df[sec_df["Sub_Dimension"].isin(sub_dims)]
        rows_box = []
        for idx, row in sec_df.iterrows():
            widget = make_value_widget(row)
            row_widgets[idx] = widget
            conf = row["Confidence"]
            caption = W.HTML(
                f"<span style='color:{_conf_color(conf)}; font-size:11px'>"
                f"conf={'' if conf is None or pd.isna(conf) else round(float(conf),2)} "
                f"src={row['Source'] or 'human'}</span>"
            )
            label = W.HTML(f"<b>{row['Sub_Dimension']}</b><br><small>{row['Field']}</small>")
            rows_box.append(W.HBox([label, widget, caption],
                                    layout=W.Layout(justify_content="space-between", margin="2px 0")))
        children.append(W.VBox(rows_box) if rows_box else W.HTML("<i>(no fields)</i>"))

    accordion.children = children
    for i, section in enumerate(sections):
        accordion.set_title(i, section)
    accordion.selected_index = 0

    status_bar.value = (f"<b>Patent {current_idx + 1}/{len(patent_ids)}</b> — {patent_id} "
                         f"{'✅ reviewed' if patent_id in done_ids else ''}")

In [ ]:
def goto(idx: int, save_current: bool = False):
    global current_idx
    if save_current:
        sync_widgets_to_df()
        pid = patent_ids[current_idx]
        rows = df[df["Patent_ID"] == pid].to_dict("records")
        now = dt.datetime.now().isoformat(timespec="seconds")
        for r in rows:
            r["Source"] = "human"
            r["Reviewer"] = REVIEWER_NAME
            r["Reviewed_At"] = now
        append_reviewed_rows(rows, REVIEWED_XLSX)
        done_ids.add(pid)

    current_idx = max(0, min(idx, len(patent_ids) - 1))
    build_form(patent_ids[current_idx])


def on_save_next(_):
    goto(current_idx + 1, save_current=True)

def on_skip(_):
    goto(current_idx + 1, save_current=False)

def on_back(_):
    sync_widgets_to_df()
    goto(current_idx - 1, save_current=False)

def on_prev_image(_):
    global current_image_idx
    sync_widgets_to_df()
    if current_image_paths:
        current_image_idx = (current_image_idx - 1) % len(current_image_paths)
    build_form(patent_ids[current_idx])

def on_next_image(_):
    global current_image_idx
    sync_widgets_to_df()
    if current_image_paths:
        current_image_idx = (current_image_idx + 1) % len(current_image_paths)
    build_form(patent_ids[current_idx])

save_btn.on_click(on_save_next)
skip_btn.on_click(on_skip)
back_btn.on_click(on_back)
img_prev_btn.on_click(on_prev_image)
img_next_btn.on_click(on_next_image)

In [ ]:
left_pane   = W.VBox([W.HTML("<h4>Patent Document</h4>"), doc_pane],
                      layout=W.Layout(width="32%", border="1px solid #ddd", padding="6px"))
middle_pane = W.VBox([W.HTML("<h4>Aircraft Display</h4>"), image_pane, image_label,
                       W.HBox([img_prev_btn, img_next_btn])],
                      layout=W.Layout(width="32%", border="1px solid #ddd", padding="6px"))
right_pane  = W.VBox([W.HTML("<h4>Taxonomy Characterization</h4>"), accordion],
                      layout=W.Layout(width="34%", border="1px solid #ddd", padding="6px"))

toolbar = W.HBox([back_btn, skip_btn, save_btn])

display(W.VBox([status_bar, W.HBox([left_pane, middle_pane, right_pane]), toolbar]))

build_form(patent_ids[current_idx])